# VCF Zarr Benchmark: polars-bio vs sgkit vs zarr-python

This notebook benchmarks VCF Zarr reads and analytical queries across:

- **polars-bio** through `scan_vcf_zarr` and DataFusion target partitions
- **sgkit** through the installed sgkit/xarray/Dask stack when the selected VCZ store is compatible
- **zarr-python** through direct raw-array reads

The benchmark is end-to-end: it can download 1000 Genomes VCF inputs, convert them to VCF Zarr with bio2zarr/vcf2zarr, inspect the resulting store, run scenarios, and plot results. Expensive download and conversion steps are gated by environment variables and never run implicitly.

The tools expose different abstraction levels. polars-bio returns a logical VCF table, sgkit/xarray works with dataset variables, and zarr-python reads raw arrays. Timed code keeps those native/optimal paths. After timing, the notebook builds small canonical summaries and fails if comparable scenarios are not semantically equivalent across tools. The canonical gate normalizes representation differences such as logical `ref`/`alt` vs raw `variant_allele`, null vs `NaN` quality values, scalar INFO strings vs one-column raw INFO arrays, and selected FORMAT/sample slices. Unsupported scenarios are recorded as skipped rows.

## Configuration

Set environment variables before running the notebook:

```bash
export VCZ_BENCH_PROFILE=30x_chr22
export VCZ_BENCH_DATA_DIR=/path/to/vcz-bench-data
export VCZ_BENCH_ENABLE_DOWNLOAD=1
export VCZ_BENCH_ENABLE_CONVERT=1
export VCZ_BENCH_FORCE_REBUILD=0
export VCZ_BENCH_TARGET_PARTITIONS=1,2,4,8,16
export VCZ_BENCH_REPETITIONS=3
export VCZ_BENCH_WARMUPS=1
export VCZ_BENCH_SAMPLE_COUNT=16
export VCZ_BENCH_SAMPLES=
export VCZ_BENCH_CONSOLIDATE_METADATA=1
```

For a prebuilt store:

```bash
export VCZ_BENCH_PROFILE=custom
export VCZ_BENCH_CUSTOM_VCZ=/path/to/data.vcz
```

For custom VCF input conversion:

```bash
export VCZ_BENCH_PROFILE=custom
export VCZ_BENCH_CUSTOM_VCF=/path/to/input.vcf.gz
export VCZ_BENCH_CUSTOM_VCZ=/path/to/output.vcz
export VCZ_BENCH_ENABLE_CONVERT=1
```

In [ ]:

from __future__ import annotations

import hashlib
import importlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import pandas as pd
import polars as pl
import polars_bio as pb

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook fallback
    display = print

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


In [ ]:

def env_flag(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    return default if value in (None, "") else int(value)


def env_csv(name: str, default: str = "") -> list[str]:
    value = os.environ.get(name, default)
    return [part.strip() for part in value.split(",") if part.strip()]


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


def optional_import(module_name: str):
    try:
        return importlib.import_module(module_name)
    except Exception as exc:
        print(f"optional dependency {module_name!r} unavailable: {exc}")
        return None


def parse_target_partitions() -> list[int]:
    values = env_csv("VCZ_BENCH_TARGET_PARTITIONS", "1,2,4,8,16")
    partitions = [int(value) for value in values]
    return sorted({max(1, value) for value in partitions})


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "polars_bio").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()


def resolve_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


PROFILE_NAME = os.environ.get("VCZ_BENCH_PROFILE", "30x_chr22")
DATA_DIR = resolve_path(os.environ.get("VCZ_BENCH_DATA_DIR", REPO_ROOT / "tmp" / "vcz-bench-data"))
ENABLE_DOWNLOAD = env_flag("VCZ_BENCH_ENABLE_DOWNLOAD", False)
ENABLE_CONVERT = env_flag("VCZ_BENCH_ENABLE_CONVERT", False)
FORCE_REBUILD = env_flag("VCZ_BENCH_FORCE_REBUILD", False)
CONSOLIDATE_METADATA = env_flag("VCZ_BENCH_CONSOLIDATE_METADATA", True)
TARGET_PARTITIONS = parse_target_partitions()
REPETITIONS = env_int("VCZ_BENCH_REPETITIONS", 3)
WARMUPS = env_int("VCZ_BENCH_WARMUPS", 1)
SAMPLE_COUNT = env_int("VCZ_BENCH_SAMPLE_COUNT", 16)
REQUESTED_SAMPLES = env_csv("VCZ_BENCH_SAMPLES", "")
REQUESTED_REGION = os.environ.get("VCZ_BENCH_REGION", "").strip()

CONFIG = {
    "profile": PROFILE_NAME,
    "repo_root": str(REPO_ROOT),
    "data_dir": str(DATA_DIR),
    "enable_download": ENABLE_DOWNLOAD,
    "enable_convert": ENABLE_CONVERT,
    "force_rebuild": FORCE_REBUILD,
    "consolidate_metadata": CONSOLIDATE_METADATA,
    "target_partitions": TARGET_PARTITIONS,
    "repetitions": REPETITIONS,
    "warmups": WARMUPS,
    "sample_count": SAMPLE_COUNT,
    "requested_samples": REQUESTED_SAMPLES,
    "requested_region": REQUESTED_REGION,
}

display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))


## Dataset Profiles

In [ ]:
FTP_BASE = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp"
PHASE3_RELEASE = f"{FTP_BASE}/release/20130502"
HIGH_COV_BASE = f"{FTP_BASE}/data_collections/1000G_2504_high_coverage/working/20220422_3202_phased_SNV_INDEL_SV"


def phase3_chr_url(chrom: int) -> str:
    return f"{PHASE3_RELEASE}/ALL.chr{chrom}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"


def high_cov_chr_url(chrom: int) -> str:
    return f"{HIGH_COV_BASE}/1kGP_high_coverage_Illumina.chr{chrom}.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"


DATASET_PROFILES: dict[str, dict[str, Any]] = {
    "phase3_chr22": {
        "description": "1000 Genomes Phase 3 chr22 genotype VCF",
        "vcf_urls": [phase3_chr_url(22)],
        "index_urls": [f"{phase3_chr_url(22)}.tbi"],
        "default_region": "22:20000000-25000000",
        "notes": "Small fallback profile; chr22 VCF is about 196 MB compressed.",
    },
    "30x_chr22": {
        "description": "1000 Genomes 30x high-coverage chr22 phased SNV/INDEL/SV panel",
        "vcf_urls": [high_cov_chr_url(22)],
        "index_urls": [f"{high_cov_chr_url(22)}.tbi"],
        "default_region": "chr22:20000000-25000000",
        "notes": "Recommended default profile; chr22 VCF is about 425 MB compressed.",
    },
    "phase3_wgs_sites": {
        "description": "1000 Genomes Phase 3 WGS sites-only VCF",
        "vcf_urls": [f"{PHASE3_RELEASE}/ALL.wgs.phase3_shapeit2_mvncall_integrated_v5c.20130502.sites.vcf.gz"],
        "index_urls": [f"{PHASE3_RELEASE}/ALL.wgs.phase3_shapeit2_mvncall_integrated_v5c.20130502.sites.vcf.gz.tbi"],
        "default_region": "22:20000000-25000000",
        "notes": "Large variant-only profile; useful when genotype/sample work is not needed.",
    },
    "phase3_wgs_autosomes": {
        "description": "1000 Genomes Phase 3 chr1-22 genotype VCFs",
        "vcf_urls": [phase3_chr_url(chrom) for chrom in range(1, 23)],
        "index_urls": [f"{phase3_chr_url(chrom)}.tbi" for chrom in range(1, 23)],
        "default_region": "22:20000000-25000000",
        "notes": "Large multi-input profile using Phase 3 autosomes.",
    },
    "30x_wgs_autosomes": {
        "description": "1000 Genomes 30x high-coverage chr1-22 phased panel",
        "vcf_urls": [high_cov_chr_url(chrom) for chrom in range(1, 23)],
        "index_urls": [f"{high_cov_chr_url(chrom)}.tbi" for chrom in range(1, 23)],
        "default_region": "chr22:20000000-25000000",
        "notes": "Stress profile using 30x high-coverage autosomes.",
    },
    "legacy_wes_exome": {
        "description": "Older 1000 Genomes exome-like genotype VCF",
        "vcf_urls": [
            f"{FTP_BASE}/technical/working/20110414_broad_exome_vcf/ALL.BI.20110228.SNPs_and_indels.exome.genotypes.vcf.gz"
        ],
        "index_urls": [
            f"{FTP_BASE}/technical/working/20110414_broad_exome_vcf/ALL.BI.20110228.SNPs_and_indels.exome.genotypes.vcf.gz.tbi"
        ],
        "default_region": "22:20000000-25000000",
        "notes": "Legacy WES-style sparse/exome workload; older and less canonical than Phase 3/30x profiles.",
    },
    "custom": {
        "description": "User-supplied VCF or VCZ paths",
        "vcf_urls": [],
        "index_urls": [],
        "default_region": "",
        "notes": "Set VCZ_BENCH_CUSTOM_VCF and/or VCZ_BENCH_CUSTOM_VCZ.",
    },
}

if PROFILE_NAME not in DATASET_PROFILES:
    available = ", ".join(sorted(DATASET_PROFILES))
    raise ValueError(f"Unknown VCZ_BENCH_PROFILE={PROFILE_NAME!r}. Available profiles: {available}")

PROFILE = DATASET_PROFILES[PROFILE_NAME]
PROFILE_DIR = DATA_DIR / PROFILE_NAME
CUSTOM_VCF = env_csv("VCZ_BENCH_CUSTOM_VCF", "")
CUSTOM_VCZ = os.environ.get("VCZ_BENCH_CUSTOM_VCZ", "").strip()

if CUSTOM_VCF:
    VCF_PATHS = [resolve_path(path) for path in CUSTOM_VCF]
    INDEX_PATHS: list[Path] = []
else:
    VCF_PATHS = [PROFILE_DIR / Path(url).name for url in PROFILE["vcf_urls"]]
    INDEX_PATHS = [PROFILE_DIR / Path(url).name for url in PROFILE["index_urls"]]

VCZ_PATH = resolve_path(CUSTOM_VCZ) if CUSTOM_VCZ else PROFILE_DIR / f"{PROFILE_NAME}.vcz"

profile_rows = []
for name, profile in DATASET_PROFILES.items():
    profile_rows.append(
        {
            "profile": name,
            "description": profile["description"],
            "inputs": len(profile["vcf_urls"]),
            "default_region": profile.get("default_region", ""),
            "notes": profile.get("notes", ""),
        }
    )

display(pd.DataFrame(profile_rows))
print(f"Selected profile: {PROFILE_NAME}")
print(f"VCZ path: {VCZ_PATH}")

## Gated Data Preparation

In [ ]:
def download_file(url: str, output: Path, force: bool = False) -> None:
    if output.exists() and not force:
        print(f"reusing {output}")
        return
    output.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["curl", "-L", "--fail", "--continue-at", "-", "--output", str(output), url],
        check=True,
    )


def run_vcf2zarr(vcf_paths: list[Path], output_vcz: Path, force: bool = False) -> None:
    if output_vcz.exists() and not force:
        print(f"reusing {output_vcz}")
        return
    exe = shutil.which("vcf2zarr")
    cmd_prefix = [exe] if exe else [sys.executable, "-m", "bio2zarr", "vcf2zarr"]
    icf_path = output_vcz.with_suffix(".icf")
    if force:
        shutil.rmtree(output_vcz, ignore_errors=True)
        shutil.rmtree(icf_path, ignore_errors=True)
    output_vcz.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([*cmd_prefix, "explode", *map(str, vcf_paths), str(icf_path)], check=True)
    subprocess.run([*cmd_prefix, "encode", str(icf_path), str(output_vcz)], check=True)


PROFILE_DIR.mkdir(parents=True, exist_ok=True)
missing_vcf_paths = [path for path in VCF_PATHS if not path.exists()]

prep_rows = [
    {"kind": "vcf", "path": str(path), "exists": path.exists()} for path in VCF_PATHS
] + [
    {"kind": "index", "path": str(path), "exists": path.exists()} for path in INDEX_PATHS
] + [
    {"kind": "vcz", "path": str(VCZ_PATH), "exists": VCZ_PATH.exists()}
]
display(pd.DataFrame(prep_rows))

if missing_vcf_paths and not ENABLE_DOWNLOAD and not VCZ_PATH.exists():
    missing = "\n".join(str(path) for path in missing_vcf_paths[:5])
    raise RuntimeError(
        "Missing VCF inputs and VCZ output is not present. "
        "Set VCZ_BENCH_ENABLE_DOWNLOAD=1, provide VCZ_BENCH_CUSTOM_VCF, "
        f"or provide VCZ_BENCH_CUSTOM_VCZ. First missing paths:\n{missing}"
    )

if ENABLE_DOWNLOAD:
    for url, path in zip(PROFILE["vcf_urls"], VCF_PATHS):
        download_file(url, path, FORCE_REBUILD)
    for url, path in zip(PROFILE["index_urls"], INDEX_PATHS):
        download_file(url, path, FORCE_REBUILD)

if not VCZ_PATH.exists():
    if not ENABLE_CONVERT:
        raise RuntimeError(
            "Missing VCZ output. Set VCZ_BENCH_ENABLE_CONVERT=1 or provide VCZ_BENCH_CUSTOM_VCZ. "
            f"Expected VCZ path: {VCZ_PATH}"
        )
    if not VCF_PATHS:
        raise RuntimeError("No VCF inputs available for conversion. Set VCZ_BENCH_CUSTOM_VCF or choose a non-custom profile.")
    run_vcf2zarr(VCF_PATHS, VCZ_PATH, FORCE_REBUILD)

print(f"Ready to benchmark VCZ store: {VCZ_PATH}")

## Optional Dependencies And Store Metadata

In [ ]:

np = optional_import("numpy")
zarr = optional_import("zarr")
sgkit = optional_import("sgkit")
xarray = optional_import("xarray")
dask = optional_import("dask")
matplotlib = optional_import("matplotlib")
seaborn = optional_import("seaborn")

ZARR_METADATA_MODE = "not_checked"
ZARR_METADATA_CONSOLIDATED = False
if CONSOLIDATE_METADATA and zarr is not None and VCZ_PATH.exists():
    try:
        zarr.consolidate_metadata(str(VCZ_PATH))
        ZARR_METADATA_MODE = "consolidated"
        ZARR_METADATA_CONSOLIDATED = True
    except Exception as exc:
        ZARR_METADATA_MODE = f"non_consolidated ({type(exc).__name__}: {exc})"
else:
    ZARR_METADATA_MODE = "non_consolidated"


def format_bytes(path: Path) -> str:
    if path.is_file():
        size = path.stat().st_size
    elif path.is_dir():
        size = sum(child.stat().st_size for child in path.rglob("*") if child.is_file())
    else:
        return "missing"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"


def decode_scalar(value: Any) -> Any:
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def decode_list(values: Any, limit: int | None = None) -> list[Any]:
    if values is None:
        return []
    if limit is not None:
        values = values[:limit]
    if hasattr(values, "tolist"):
        values = values.tolist()
    return [decode_scalar(value) for value in values]


def open_zarr_group(path: Path):
    if zarr is None:
        raise RuntimeError("zarr-python is not installed")
    return zarr.open_group(str(path), mode="r")


def zarr_array_keys(group) -> list[str]:
    try:
        return sorted(group.array_keys())
    except Exception:
        keys = []
        for key in group.keys():
            try:
                obj = group[key]
                if hasattr(obj, "shape"):
                    keys.append(key)
            except Exception:
                continue
        return sorted(keys)


try:
    ZARR_GROUP = open_zarr_group(VCZ_PATH)
    ARRAY_NAMES = zarr_array_keys(ZARR_GROUP)
except Exception as exc:
    print(f"zarr metadata unavailable: {exc}")
    ZARR_GROUP = None
    ARRAY_NAMES = []


def has_array(name: str) -> bool:
    return ZARR_GROUP is not None and name in ARRAY_NAMES


def zarr_array(name: str):
    if not has_array(name):
        raise SkipBenchmark(f"VCZ array {name!r} is not available")
    return ZARR_GROUP[name]


def read_zarr(name: str, selection: Any = slice(None)):
    return zarr_array(name)[selection]


def first_n(name: str, n: int = 10) -> list[Any]:
    if not has_array(name):
        return []
    arr = zarr_array(name)
    size = arr.shape[0] if arr.shape else 0
    return decode_list(arr[: min(size, n)])


def infer_variant_count() -> int | None:
    if has_array("variant_position"):
        return int(zarr_array("variant_position").shape[0])
    try:
        df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(pl.len().alias("n")).collect()
        return int(df["n"][0])
    except Exception:
        return None


def infer_sample_ids(limit: int | None = None) -> list[str]:
    if not has_array("sample_id"):
        return []
    try:
        arr = zarr_array("sample_id")
        count = arr.shape[0] if arr.shape else 0
        return [str(value) for value in decode_list(arr[: count if limit is None else min(count, limit)])]
    except Exception:
        return []


def contig_lookup() -> dict[int, str]:
    if not has_array("contig_id"):
        return {}
    values = first_n("contig_id", zarr_array("contig_id").shape[0])
    return {idx: str(value) for idx, value in enumerate(values)}


def parse_region(region: str) -> tuple[str, int, int] | None:
    if not region:
        return None
    match = re.match(r"^([^:]+):(\d+)-(\d+)$", region.replace(",", ""))
    if not match:
        raise ValueError(f"Region must look like chrom:start-end, got {region!r}")
    chrom, start, end = match.groups()
    return chrom, int(start), int(end)


def infer_region() -> tuple[str, int, int]:
    explicit = parse_region(REQUESTED_REGION)
    if explicit is not None:
        return explicit
    profile_region = parse_region(PROFILE.get("default_region", ""))
    if PROFILE_NAME != "custom" and profile_region is not None:
        return profile_region
    if has_array("variant_position"):
        positions = read_zarr("variant_position", slice(0, min(1000, zarr_array("variant_position").shape[0])))
        pos_list = decode_list(positions)
        if pos_list:
            start = int(pos_list[0])
            end = start + max(10_000, min(1_000_000, max(pos_list) - start if len(pos_list) > 1 else 10_000))
            lookup = contig_lookup()
            chrom = ""
            if has_array("variant_contig"):
                contig_values = decode_list(read_zarr("variant_contig", slice(0, 1)))
                if contig_values:
                    chrom = lookup.get(int(contig_values[0]), str(contig_values[0]))
            return chrom or "", start, end
    df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(["chrom", "start"]).head(1).collect()
    return str(df["chrom"][0]), int(df["start"][0]), int(df["start"][0]) + 10_000


def choose_info_field() -> str | None:
    for candidate in ["AF", "DP", "AC", "AN"]:
        if has_array(f"variant_{candidate}"):
            return candidate
    for name in ARRAY_NAMES:
        if name.startswith("variant_") and name not in {
            "variant_contig",
            "variant_position",
            "variant_length",
            "variant_allele",
            "variant_id",
            "variant_id_mask",
            "variant_filter",
            "variant_quality",
        }:
            return name.removeprefix("variant_")
    return None


def choose_format_field() -> str | None:
    if has_array("call_GT") or has_array("call_genotype"):
        return "GT"
    for candidate in ["DP", "GQ"]:
        if has_array(f"call_{candidate}"):
            return candidate
    for name in ARRAY_NAMES:
        if name.startswith("call_") and not name.endswith("_mask") and not name.endswith("_phased"):
            return name.removeprefix("call_")
    return None


def format_array_name(field_id: str) -> str:
    if field_id == "GT" and has_array("call_genotype"):
        return "call_genotype"
    return f"call_{field_id}"


VARIANT_COUNT = infer_variant_count()
SAMPLE_IDS = infer_sample_ids()
SAMPLE_ID_TO_INDEX = {sample: idx for idx, sample in enumerate(SAMPLE_IDS)}
SAMPLES_FOR_QUERY = REQUESTED_SAMPLES or SAMPLE_IDS[:SAMPLE_COUNT]
BENCH_REGION = infer_region()
INFO_FIELD = choose_info_field()
FORMAT_FIELD = choose_format_field()
CONTIG_LOOKUP = contig_lookup()
CONTIG_NAME_TO_INDEX = {name: idx for idx, name in CONTIG_LOOKUP.items()}

metadata = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "polars_bio": package_version("polars-bio"),
    "polars": package_version("polars"),
    "pandas": package_version("pandas"),
    "sgkit": package_version("sgkit"),
    "zarr": package_version("zarr"),
    "xarray": package_version("xarray"),
    "dask": package_version("dask"),
    "profile": PROFILE_NAME,
    "vcz_path": str(VCZ_PATH),
    "vcz_size": format_bytes(VCZ_PATH),
    "zarr_metadata": ZARR_METADATA_MODE,
    "variant_count": VARIANT_COUNT,
    "sample_count": len(SAMPLE_IDS),
    "array_count": len(ARRAY_NAMES),
    "selected_region": f"{BENCH_REGION[0]}:{BENCH_REGION[1]}-{BENCH_REGION[2]}",
    "info_field": INFO_FIELD,
    "format_field": FORMAT_FIELD,
    "sample_query_count": len(SAMPLES_FOR_QUERY),
}

display(pd.DataFrame([metadata]).T.rename(columns={0: "value"}))
if ARRAY_NAMES:
    display(pd.DataFrame({"array": ARRAY_NAMES, "shape": [str(zarr_array(name).shape) for name in ARRAY_NAMES]}).head(80))


## Benchmark Harness

In [ ]:

class SkipBenchmark(Exception):
    pass


@dataclass
class ScenarioOutput:
    native: Any
    canonical: dict[str, Any]
    validation: str = ""


@dataclass
class ResultRow:
    tool: str
    scenario: str
    profile: str
    partition: str
    repetition: int
    status: str
    elapsed_s: float | None
    validation: str
    canonical: str
    error: str
    notes: str


def jsonable(value: Any) -> Any:
    value = decode_scalar(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in sorted(value.items(), key=lambda item: str(item[0]))}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        return jsonable(value.tolist())
    return value


def canonical_json(value: dict[str, Any]) -> str:
    return json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"))


def digest_values(values: Any) -> str:
    h = hashlib.sha256()
    if np is not None:
        arr = np.asarray(values)
        h.update(str(arr.shape).encode())
        h.update(str(arr.dtype).encode())
        if arr.dtype.kind in {"O", "U", "S"}:
            for value in arr.ravel(order="C"):
                h.update(str(decode_scalar(value)).encode())
                h.update(b"\0")
        else:
            h.update(np.ascontiguousarray(arr).tobytes())
    else:
        for value in decode_list(values):
            h.update(str(value).encode())
            h.update(b"\0")
    return h.hexdigest()


def contig_names_from_codes(values: Any) -> list[str]:
    return [CONTIG_LOOKUP.get(int(value), str(value)) for value in decode_list(values)]


def digest_canonical_rows(rows: Any) -> str:
    # Order-independent multiset digest: enough for the quality gate, without sorting million-row scans.
    count = 0
    xor_digest = 0
    sum_digest = 0
    modulus = 1 << 256
    for row in rows:
        digest = hashlib.sha256(canonical_json({"row": row}).encode()).digest()
        value = int.from_bytes(digest, "big")
        count += 1
        xor_digest ^= value
        sum_digest = (sum_digest + value) % modulus
    return f"{count}:{xor_digest:064x}:{sum_digest:064x}"


def variant_key_sort_tuple(row: list[Any]) -> tuple[str, int]:
    return str(row[0]), int(row[1])


def variant_keys(chroms: Any, positions: Any) -> list[list[Any]]:
    chrom_list = [str(value) for value in decode_list(chroms)]
    position_list = [int(value) for value in decode_list(positions)]
    return [[chrom, position] for chrom, position in zip(chrom_list, position_list)]


def variant_payload_records(chroms: Any, positions: Any, payload_rows: Any) -> list[list[Any]]:
    records = []
    for key, payload in zip(variant_keys(chroms, positions), payload_rows):
        records.append([key[0], key[1], payload])
    return records


def semantic_variants(chroms: Any, positions: Any, **extra: Any) -> dict[str, Any]:
    rows = variant_keys(chroms, positions)
    chrom_list = [row[0] for row in rows]
    position_list = [row[1] for row in rows]
    counts = dict(pd.Series(chrom_list).value_counts().sort_index()) if chrom_list else {}
    canonical = {
        "rows": len(position_list),
        "chrom_counts": {str(key): int(value) for key, value in counts.items()},
        "first": min(rows, key=variant_key_sort_tuple) if rows else None,
        "last": max(rows, key=variant_key_sort_tuple) if rows else None,
        "position_sum": int(sum(position_list)),
        "position_digest": digest_canonical_rows(rows),
    }
    canonical.update(extra)
    return canonical


def is_missing_scalar(value: Any) -> bool:
    value = decode_scalar(value)
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() in {"", ".", "nan", "NaN"}
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    return bool(missing)


def normalize_numeric_or_text(value: Any) -> Any:
    value = decode_scalar(value)
    if is_missing_scalar(value):
        return None
    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, int):
        return int(value)
    if isinstance(value, float):
        return format(float(value), ".7g")
    if isinstance(value, str):
        text = value.strip()
        if "," in text:
            return normalize_value_sequence(text.split(","))
        try:
            return format(float(text), ".7g")
        except ValueError:
            return text
    return str(value)


def normalize_value_sequence(values: Any) -> Any:
    items = [normalize_value(value) for value in decode_list(values)]
    while items and items[-1] is None:
        items.pop()
    if not items:
        return None
    if len(items) == 1:
        return items[0]
    return items


def normalize_value(value: Any) -> Any:
    value = decode_scalar(value)
    if isinstance(value, (list, tuple)):
        return normalize_value_sequence(value)
    if np is not None and hasattr(value, "tolist") and not np.isscalar(value):
        return normalize_value_sequence(value.tolist())
    return normalize_numeric_or_text(value)


def is_nested_value(value: Any) -> bool:
    value = decode_scalar(value)
    return isinstance(value, (list, tuple)) or (np is not None and hasattr(value, "tolist") and not np.isscalar(value))


def normalize_value_rows(values: Any) -> list[Any]:
    if hasattr(values, "tolist"):
        values = values.tolist()
    values = decode_scalar(values)
    if isinstance(values, (list, tuple)):
        if any(is_nested_value(value) for value in values):
            return [normalize_value_sequence(value) if is_nested_value(value) else normalize_value(value) for value in values]
        return [normalize_value(value) for value in values]
    return [normalize_value(values)]


def normalized_non_missing_count(rows: list[Any]) -> int:
    return sum(row is not None for row in rows)


def split_alleles(value: Any) -> list[str]:
    value = decode_scalar(value)
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(decode_scalar(item)) for item in value if not is_missing_scalar(item)]
    if np is not None and hasattr(value, "tolist") and not np.isscalar(value):
        return split_alleles(value.tolist())
    text = str(value).strip()
    if not text or text == ".":
        return []
    return [part.strip() for part in text.split(",") if part.strip() and part.strip() != "."]


def allele_rows_from_df(df: pl.DataFrame) -> list[list[str]]:
    if not {"ref", "alt"}.issubset(df.columns):
        return []
    return [[str(decode_scalar(ref)), *split_alleles(alt)] for ref, alt in zip(df["ref"].to_list(), df["alt"].to_list())]


def allele_rows_from_array(alleles: Any) -> list[list[str]]:
    rows = []
    for row in decode_list(alleles):
        rows.append(split_alleles(row))
    return rows


def allele_digest_from_df(df: pl.DataFrame) -> str:
    return digest_canonical_rows(
        variant_payload_records(df["chrom"].to_list(), df["start"].to_list(), allele_rows_from_df(df))
    )


def allele_digest_from_array(chroms: Any, positions: Any, alleles: Any) -> str:
    return digest_canonical_rows(variant_payload_records(chroms, positions, allele_rows_from_array(alleles)))


def add_projected_value(canonical: dict[str, Any], chroms: Any, positions: Any, values: Any) -> None:
    rows = normalize_value_rows(values)
    canonical["value_digest"] = digest_canonical_rows(variant_payload_records(chroms, positions, rows))
    canonical["value_non_missing"] = normalized_non_missing_count(rows)


def canonical_from_pb_df(df: pl.DataFrame, include_alleles: bool = False, value_column: str | None = None) -> dict[str, Any]:
    chroms = df["chrom"].to_list()
    positions = df["start"].to_list()
    canonical = semantic_variants(chroms, positions)
    if include_alleles:
        canonical["allele_digest"] = allele_digest_from_df(df)
    if value_column and value_column in df.columns:
        add_projected_value(canonical, chroms, positions, df[value_column].to_list())
    return canonical


def canonical_from_raw_arrays(payload: dict[str, Any], include_alleles: bool = False, value_name: str | None = None) -> dict[str, Any]:
    chroms = payload["chrom"] if "chrom" in payload else contig_names_from_codes(payload["contig"])
    positions = payload["position"]
    canonical = semantic_variants(chroms, positions)
    if include_alleles and "allele" in payload:
        canonical["allele_digest"] = allele_digest_from_array(chroms, positions, payload["allele"])
    if value_name and "value" in payload:
        add_projected_value(canonical, chroms, positions, payload["value"])
    return canonical


def canonical_from_sgkit_dataset(ds: Any, include_alleles: bool = False, value_name: str | None = None) -> dict[str, Any]:
    contig_values = ds["variant_contig"].values if "variant_contig" in ds else []
    position_values = ds["variant_position"].values if "variant_position" in ds else []
    chroms = contig_names_from_codes(contig_values)
    canonical = semantic_variants(chroms, position_values)
    if include_alleles and "variant_allele" in ds:
        canonical["allele_digest"] = allele_digest_from_array(chroms, position_values, ds["variant_allele"].values)
    if value_name and value_name in ds:
        add_projected_value(canonical, chroms, position_values, ds[value_name].values)
    return canonical


def format_gt_values(values: Any, phased: bool) -> str:
    sep = "|" if bool(phased) else "/"
    return sep.join("." if int(value) < 0 else str(int(value)) for value in decode_list(values))


def sample_values_from_pb_item(item: Any, field: str) -> list[str]:
    if item is None:
        return []
    if hasattr(item, "as_py"):
        item = item.as_py()
    if not isinstance(item, dict):
        return []
    values = item.get(field) or item.get("GT") or item.get("genotype") or []
    return [str(value) for value in values]


def canonical_sample_strings_from_pb(df: pl.DataFrame, field: str) -> dict[str, Any]:
    row_values = []
    sample_count = 0
    for item in df["genotypes"].to_list():
        values = sample_values_from_pb_item(item, field)
        sample_count = max(sample_count, len(values))
        row_values.append(values)
    chroms = df["chrom"].to_list()
    positions = df["start"].to_list()
    return {
        **canonical_from_pb_df(df),
        "samples": sample_count,
        "format_field": field,
        "format_digest": digest_canonical_rows(variant_payload_records(chroms, positions, row_values)),
    }


def raw_format_rows(payload: dict[str, Any], field: str) -> tuple[list[list[str]], int]:
    gt = payload["value"]
    phased = payload.get("phased")
    if np is None:
        rows = [[str(decode_scalar(value))] for value in decode_list(gt)]
        return rows, len(SAMPLES_FOR_QUERY)
    gt_arr = np.asarray(gt)
    sample_count = int(gt_arr.shape[1]) if gt_arr.ndim > 1 else len(SAMPLES_FOR_QUERY)
    rows = []
    if field == "GT" and phased is not None:
        phased_arr = np.asarray(phased)
        for row in range(gt_arr.shape[0]):
            row_values = []
            for sample in range(gt_arr.shape[1]):
                row_values.append(format_gt_values(gt_arr[row, sample], bool(phased_arr[row, sample])))
            rows.append(row_values)
    elif gt_arr.ndim <= 1:
        rows = [[str(decode_scalar(value))] for value in gt_arr.tolist()]
    else:
        for row in range(gt_arr.shape[0]):
            rows.append([str(decode_scalar(value)) for value in np.asarray(gt_arr[row]).ravel(order="C")])
    return rows, sample_count


def canonical_sample_strings_from_raw(payload: dict[str, Any], field: str) -> dict[str, Any]:
    chroms = payload["chrom"] if "chrom" in payload else contig_names_from_codes(payload["contig"])
    positions = payload["position"]
    rows, sample_count = raw_format_rows(payload, field)
    return {
        **canonical_from_raw_arrays(payload),
        "samples": sample_count,
        "format_field": field,
        "format_digest": digest_canonical_rows(variant_payload_records(chroms, positions, rows)),
    }


def validate_any(native: Any, canonical: dict[str, Any] | None = None) -> str:
    if canonical:
        parts = []
        for key in ["rows", "variants", "samples", "arrays", "columns", "format_field"]:
            if key in canonical:
                parts.append(f"{key}={canonical[key]}")
        if parts:
            return ", ".join(parts)
    if isinstance(native, pl.DataFrame):
        return f"rows={native.height}, cols={len(native.columns)}"
    if isinstance(native, pd.DataFrame):
        return f"rows={len(native)}, cols={len(native.columns)}"
    if isinstance(native, dict):
        return ", ".join(f"{key}={native[key]}" for key in ["rows", "variants", "samples", "shape", "arrays", "columns"] if key in native) or "dict"
    if hasattr(native, "sizes"):
        return ", ".join(f"{k}={v}" for k, v in dict(native.sizes).items())
    if hasattr(native, "shape"):
        return f"shape={native.shape}"
    return type(native).__name__


def measure(
    tool: str,
    scenario: str,
    partition: str,
    fn: Callable[[], Any],
    canonicalize: Callable[[Any], dict[str, Any]],
    notes: str = "",
) -> list[ResultRow]:
    rows: list[ResultRow] = []
    canonical: dict[str, Any] | None = None
    canonical_text = ""
    try:
        for _ in range(WARMUPS):
            fn()
        for rep in range(REPETITIONS):
            started = time.perf_counter()
            native = fn()
            elapsed = time.perf_counter() - started
            if canonical is None:
                canonical = canonicalize(native)
                canonical_text = canonical_json(canonical)
            rows.append(
                ResultRow(
                    tool=tool,
                    scenario=scenario,
                    profile=PROFILE_NAME,
                    partition=partition,
                    repetition=rep,
                    status="ok",
                    elapsed_s=elapsed,
                    validation=validate_any(native, canonical),
                    canonical=canonical_text,
                    error="",
                    notes=notes,
                )
            )
    except SkipBenchmark as exc:
        rows.append(ResultRow(tool, scenario, PROFILE_NAME, partition, -1, "skipped", None, "", "", str(exc), notes))
    except Exception as exc:
        rows.append(ResultRow(tool, scenario, PROFILE_NAME, partition, -1, "failed", None, "", "", repr(exc), notes))
    return rows


def core_columns() -> list[str]:
    return ["chrom", "start", "end", "ref", "alt"]


def available_core_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        names = set(lf.collect_schema().names())
    except Exception:
        names = set(lf.columns)
    return [column for column in core_columns() if column in names]


def sampled_variants(limit: int = 5) -> list[tuple[str, int]]:
    if has_array("variant_position"):
        positions = [int(value) for value in decode_list(read_zarr("variant_position", slice(0, min(limit, zarr_array("variant_position").shape[0]))))]
        chrom = BENCH_REGION[0]
        if has_array("variant_contig"):
            contigs = decode_list(read_zarr("variant_contig", slice(0, len(positions))))
            return [(CONTIG_LOOKUP.get(int(contig), str(contig)), pos) for contig, pos in zip(contigs, positions)]
        return [(chrom, pos) for pos in positions]
    df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(["chrom", "start"]).head(limit).collect()
    return [(str(chrom), int(pos)) for chrom, pos in zip(df["chrom"].to_list(), df["start"].to_list())]


LOOKUP_VARIANTS = sampled_variants()
LOOKUP_VARIANT = LOOKUP_VARIANTS[0] if LOOKUP_VARIANTS else BENCH_REGION[:2]
print(f"Lookup variant: {LOOKUP_VARIANT}")
print(f"Benchmark region: {BENCH_REGION}")


## Tool Adapters And Scenarios

In [ ]:

def pb_scan(partitions: int, **kwargs) -> pl.LazyFrame:
    pb.set_option("datafusion.execution.target_partitions", str(partitions))
    return pb.scan_vcf_zarr(str(VCZ_PATH), **kwargs)


def pb_open_metadata(partitions: int) -> dict[str, Any]:
    lf = pb_scan(partitions)
    schema = lf.collect_schema()
    return {"columns": len(schema.names()), "arrays": len(ARRAY_NAMES), "variants": VARIANT_COUNT, "samples": len(SAMPLE_IDS)}


def pb_full_core(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    return lf.select(available_core_columns(lf)).collect()


def pb_projection_min(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    cols = [column for column in ["chrom", "start"] if column in set(lf.collect_schema().names())]
    return lf.select(cols).collect()


def pb_projection_quality(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    cols = [column for column in ["chrom", "start", "qual"] if column in set(lf.collect_schema().names())]
    if len(cols) < 3:
        raise SkipBenchmark("qual column is not available")
    return lf.select(cols).collect()


def pb_info_projection(partitions: int) -> pl.DataFrame:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    lf = pb_scan(partitions, info_fields=[INFO_FIELD])
    return lf.select(["chrom", "start", INFO_FIELD]).collect()


def pb_range_filter(partitions: int) -> pl.DataFrame:
    chrom, start, end = BENCH_REGION
    lf = pb_scan(partitions)
    cols = available_core_columns(lf)
    return (
        lf.filter((pl.col("chrom") == chrom) & (pl.col("start") >= start) & (pl.col("start") <= end))
        .select(cols)
        .collect()
    )


def pb_variant_lookup(partitions: int) -> pl.DataFrame:
    chrom, position = LOOKUP_VARIANT
    lf = pb_scan(partitions)
    cols = available_core_columns(lf)
    return lf.filter((pl.col("chrom") == chrom) & (pl.col("start") == position)).select(cols).collect()


def pb_sample_format(partitions: int) -> pl.DataFrame:
    if FORMAT_FIELD is None:
        raise SkipBenchmark("no FORMAT field array found")
    if not SAMPLES_FOR_QUERY:
        raise SkipBenchmark("no sample IDs available")
    chrom, start, end = BENCH_REGION
    lf = pb_scan(partitions, format_fields=[FORMAT_FIELD], samples=SAMPLES_FOR_QUERY)
    return (
        lf.filter((pl.col("chrom") == chrom) & (pl.col("start") >= start) & (pl.col("start") <= end))
        .select(["chrom", "start", "genotypes"])
        .collect()
    )


def pb_count_by_chrom(partitions: int) -> pl.DataFrame:
    return pb_scan(partitions).group_by("chrom").agg(pl.len().alias("variants")).collect()


def zarr_open_metadata() -> dict[str, Any]:
    if ZARR_GROUP is None:
        raise SkipBenchmark("zarr-python could not open the VCZ store")
    return {"arrays": len(ARRAY_NAMES), "variants": VARIANT_COUNT, "samples": len(SAMPLE_IDS)}


def raw_core_payload(selection: Any = slice(None), include_alleles: bool = False, value_name: str | None = None) -> dict[str, Any]:
    payload = {
        "contig": read_zarr("variant_contig", selection),
        "position": read_zarr("variant_position", selection),
    }
    if include_alleles:
        payload["allele"] = read_zarr("variant_allele", selection)
    if value_name:
        payload["value"] = read_zarr(value_name, selection)
    return payload


def zarr_full_core() -> dict[str, Any]:
    for name in ["variant_contig", "variant_position", "variant_allele"]:
        if not has_array(name):
            raise SkipBenchmark(f"required raw array {name!r} is missing")
    return raw_core_payload(include_alleles=True)


def zarr_projection_min() -> dict[str, Any]:
    return raw_core_payload()


def zarr_projection_quality() -> dict[str, Any]:
    return raw_core_payload(value_name="variant_quality")


def zarr_info_projection() -> dict[str, Any]:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    return raw_core_payload(value_name=f"variant_{INFO_FIELD}")


def region_indexes() -> Any:
    chrom, start, end = BENCH_REGION
    positions = read_zarr("variant_position")
    if has_array("variant_contig") and chrom in CONTIG_NAME_TO_INDEX:
        contigs = read_zarr("variant_contig")
        chrom_index = CONTIG_NAME_TO_INDEX[chrom]
        if np is not None:
            return np.flatnonzero((contigs == chrom_index) & (positions >= start) & (positions <= end))
        return [idx for idx, (contig, pos) in enumerate(zip(contigs, positions)) if int(contig) == chrom_index and start <= int(pos) <= end]
    if np is not None:
        return np.flatnonzero((positions >= start) & (positions <= end))
    return [idx for idx, pos in enumerate(positions) if start <= int(pos) <= end]


def contiguous_selection(indexes: Any) -> slice:
    values = indexes.tolist() if hasattr(indexes, "tolist") else list(indexes)
    if not values:
        return slice(0, 0)
    return slice(int(values[0]), int(values[-1]) + 1)


def zarr_range_filter() -> dict[str, Any]:
    indexes = region_indexes()
    return raw_core_payload(contiguous_selection(indexes), include_alleles=True)


def zarr_variant_lookup() -> dict[str, Any]:
    chrom, position = LOOKUP_VARIANT
    positions = read_zarr("variant_position")
    if has_array("variant_contig") and chrom in CONTIG_NAME_TO_INDEX:
        contigs = read_zarr("variant_contig")
        chrom_index = CONTIG_NAME_TO_INDEX[chrom]
        if np is not None:
            indexes = np.flatnonzero((contigs == chrom_index) & (positions == position))
        else:
            indexes = [idx for idx, (contig, pos) in enumerate(zip(contigs, positions)) if int(contig) == chrom_index and int(pos) == int(position)]
    elif np is not None:
        indexes = np.flatnonzero(positions == position)
    else:
        indexes = [idx for idx, pos in enumerate(positions) if int(pos) == int(position)]
    return raw_core_payload(contiguous_selection(indexes), include_alleles=True)


def zarr_sample_format() -> dict[str, Any]:
    if FORMAT_FIELD is None:
        raise SkipBenchmark("no FORMAT field array found")
    name = format_array_name(FORMAT_FIELD)
    arr = zarr_array(name)
    if not SAMPLES_FOR_QUERY:
        raise SkipBenchmark("no sample IDs available")
    sample_indexes = [SAMPLE_ID_TO_INDEX[sample] for sample in SAMPLES_FOR_QUERY if sample in SAMPLE_ID_TO_INDEX]
    if not sample_indexes:
        raise SkipBenchmark("requested samples are not present")
    row_selection = contiguous_selection(region_indexes())
    sample_start = min(sample_indexes)
    sample_stop = max(sample_indexes) + 1
    selection = (row_selection, slice(sample_start, sample_stop))
    if len(arr.shape) > 2:
        selection = (*selection, *[slice(None) for _ in arr.shape[2:]])
    data = arr[selection]
    if np is not None:
        selected = np.asarray(data)[:, [idx - sample_start for idx in sample_indexes], ...]
    else:
        selected = data
    payload = {**raw_core_payload(row_selection), "value": selected}
    if FORMAT_FIELD == "GT" and has_array("call_genotype_phased"):
        phased = zarr_array("call_genotype_phased")[(row_selection, slice(sample_start, sample_stop))]
        if np is not None:
            phased = np.asarray(phased)[:, [idx - sample_start for idx in sample_indexes]]
        payload["phased"] = phased
    return payload


def zarr_count_by_chrom() -> dict[str, Any]:
    if not has_array("variant_contig"):
        raise SkipBenchmark("variant_contig is missing")
    contigs = contig_names_from_codes(read_zarr("variant_contig"))
    counts = pd.Series(contigs).value_counts().sort_index()
    return {"counts": {str(key): int(value) for key, value in counts.items()}}


SGKIT_DATASET_CACHE = None


def open_sgkit_dataset(use_cache: bool = True):
    global SGKIT_DATASET_CACHE
    if use_cache and SGKIT_DATASET_CACHE is not None:
        return SGKIT_DATASET_CACHE
    kwargs = {"consolidated": ZARR_METADATA_CONSOLIDATED}
    errors = []
    if sgkit is not None and hasattr(sgkit, "load_dataset"):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                ds = sgkit.load_dataset(str(VCZ_PATH), **kwargs)
            if use_cache:
                SGKIT_DATASET_CACHE = ds
            return ds
        except Exception as exc:
            errors.append(f"sgkit.load_dataset: {exc}")
    if xarray is not None and hasattr(xarray, "open_zarr"):
        try:
            ds = xarray.open_zarr(str(VCZ_PATH), consolidated=ZARR_METADATA_CONSOLIDATED)
            if use_cache:
                SGKIT_DATASET_CACHE = ds
            return ds
        except Exception as exc:
            errors.append(f"xarray.open_zarr: {exc}")
    raise SkipBenchmark("; ".join(errors) or "sgkit/xarray are not installed")


def sgkit_vars(ds) -> set[str]:
    return set(getattr(ds, "data_vars", {}).keys())


def sgkit_require(ds, names: list[str]) -> None:
    available = sgkit_vars(ds)
    missing = [name for name in names if name not in available]
    if missing:
        raise SkipBenchmark(f"sgkit/xarray dataset missing variables: {missing}")


def sgkit_open_metadata() -> dict[str, Any]:
    ds = open_sgkit_dataset(use_cache=False)
    return {"arrays": len(sgkit_vars(ds)), "variants": int(ds.sizes.get("variants", 0)), "samples": int(ds.sizes.get("samples", 0))}


def sgkit_load_vars(names: list[str]) -> Any:
    ds = open_sgkit_dataset()
    sgkit_require(ds, names)
    return ds[names].load()


def sgkit_full_core() -> Any:
    return sgkit_load_vars(["variant_contig", "variant_position", "variant_allele"])


def sgkit_projection_min() -> Any:
    return sgkit_load_vars(["variant_contig", "variant_position"])


def sgkit_projection_quality() -> Any:
    return sgkit_load_vars(["variant_contig", "variant_position", "variant_quality"])


def sgkit_info_projection() -> Any:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    return sgkit_load_vars(["variant_contig", "variant_position", f"variant_{INFO_FIELD}"])


def sgkit_region_mask(ds: Any) -> Any:
    sgkit_require(ds, ["variant_contig", "variant_position"])
    chrom, start, end = BENCH_REGION
    positions = ds["variant_position"].load()
    contigs = ds["variant_contig"].load()
    if chrom in CONTIG_NAME_TO_INDEX:
        return (contigs == CONTIG_NAME_TO_INDEX[chrom]) & (positions >= start) & (positions <= end)
    return (positions >= start) & (positions <= end)


def sgkit_range_filter() -> Any:
    ds = open_sgkit_dataset()
    mask = sgkit_region_mask(ds)
    names = [name for name in ["variant_contig", "variant_position", "variant_allele"] if name in ds]
    return ds[names].isel(variants=mask).load()


def sgkit_variant_lookup() -> Any:
    ds = open_sgkit_dataset()
    sgkit_require(ds, ["variant_contig", "variant_position"])
    chrom, position = LOOKUP_VARIANT
    positions = ds["variant_position"].load()
    contigs = ds["variant_contig"].load()
    if chrom in CONTIG_NAME_TO_INDEX:
        mask = (contigs == CONTIG_NAME_TO_INDEX[chrom]) & (positions == position)
    else:
        mask = positions == position
    names = [name for name in ["variant_contig", "variant_position", "variant_allele"] if name in ds]
    return ds[names].isel(variants=mask).load()


def sgkit_sample_format() -> Any:
    ds = open_sgkit_dataset()
    _chrom, _start, _end = BENCH_REGION
    data_var = format_array_name(FORMAT_FIELD or "")
    if data_var not in sgkit_vars(ds):
        raise SkipBenchmark(f"no compatible call field found for {FORMAT_FIELD}")
    if "samples" not in ds.sizes:
        raise SkipBenchmark("dataset has no samples dimension")
    sample_indexes = [SAMPLE_ID_TO_INDEX[sample] for sample in SAMPLES_FOR_QUERY if sample in SAMPLE_ID_TO_INDEX]
    if not sample_indexes:
        raise SkipBenchmark("requested samples are not present")
    mask = sgkit_region_mask(ds)
    names = ["variant_contig", "variant_position", data_var]
    if FORMAT_FIELD == "GT" and "call_genotype_phased" in ds:
        names.append("call_genotype_phased")
    return ds[names].isel(variants=mask, samples=sample_indexes).load()


def sgkit_count_by_chrom() -> Any:
    return sgkit_load_vars(["variant_contig"])


## Run Benchmarks

In [ ]:

def canonical_metadata(native: dict[str, Any]) -> dict[str, Any]:
    return {"variants": int(native.get("variants") or VARIANT_COUNT or 0), "samples": int(native.get("samples") or len(SAMPLE_IDS)), "arrays": int(native.get("arrays") or len(ARRAY_NAMES))}


def canonical_count_map(native: Any) -> dict[str, Any]:
    if isinstance(native, pl.DataFrame):
        counts = {str(row[0]): int(row[1]) for row in native.sort("chrom").iter_rows()}
    elif isinstance(native, dict):
        counts = {str(key): int(value) for key, value in native["counts"].items()}
    else:
        contigs = contig_names_from_codes(native["variant_contig"].values)
        counts = {str(key): int(value) for key, value in pd.Series(contigs).value_counts().sort_index().items()}
    return {"chrom_counts": counts, "variants": int(sum(counts.values())), "rows": len(counts)}


def canonical_pb_full(native: pl.DataFrame) -> dict[str, Any]:
    return canonical_from_pb_df(native, include_alleles=True)


def canonical_pb_min(native: pl.DataFrame) -> dict[str, Any]:
    return canonical_from_pb_df(native)


def canonical_pb_quality(native: pl.DataFrame) -> dict[str, Any]:
    return canonical_from_pb_df(native, value_column="qual")


def canonical_pb_info(native: pl.DataFrame) -> dict[str, Any]:
    return canonical_from_pb_df(native, value_column=INFO_FIELD)


def canonical_pb_sample(native: pl.DataFrame) -> dict[str, Any]:
    return canonical_sample_strings_from_pb(native, FORMAT_FIELD or "")


def canonical_raw_full(native: dict[str, Any]) -> dict[str, Any]:
    return canonical_from_raw_arrays(native, include_alleles=True)


def canonical_raw_min(native: dict[str, Any]) -> dict[str, Any]:
    return canonical_from_raw_arrays(native)


def canonical_raw_quality(native: dict[str, Any]) -> dict[str, Any]:
    return canonical_from_raw_arrays(native, value_name="variant_quality")


def canonical_raw_info(native: dict[str, Any]) -> dict[str, Any]:
    return canonical_from_raw_arrays(native, value_name=f"variant_{INFO_FIELD}")


def canonical_raw_sample(native: dict[str, Any]) -> dict[str, Any]:
    return canonical_sample_strings_from_raw(native, FORMAT_FIELD or "")


def canonical_sgkit_full(native: Any) -> dict[str, Any]:
    return canonical_from_sgkit_dataset(native, include_alleles=True)


def canonical_sgkit_min(native: Any) -> dict[str, Any]:
    return canonical_from_sgkit_dataset(native)


def canonical_sgkit_quality(native: Any) -> dict[str, Any]:
    return canonical_from_sgkit_dataset(native, value_name="variant_quality")


def canonical_sgkit_info(native: Any) -> dict[str, Any]:
    return canonical_from_sgkit_dataset(native, value_name=f"variant_{INFO_FIELD}")


def canonical_sgkit_sample(native: Any) -> dict[str, Any]:
    data_var = format_array_name(FORMAT_FIELD or "")
    payload = {
        "contig": native["variant_contig"].values,
        "position": native["variant_position"].values,
        "value": native[data_var].values,
    }
    if FORMAT_FIELD == "GT" and "call_genotype_phased" in native:
        payload["phased"] = native["call_genotype_phased"].values
    return canonical_sample_strings_from_raw(payload, FORMAT_FIELD or "")


POLARS_SCENARIOS: list[tuple[str, Callable[[int], Any], Callable[[Any], dict[str, Any]], str]] = [
    ("open_metadata", pb_open_metadata, canonical_metadata, "DataFusion schema creation; no row materialization"),
    ("full_core_scan", pb_full_core, canonical_pb_full, "Logical VCF core columns"),
    ("projection_chrom_start", pb_projection_min, canonical_pb_min, "Projection pruning to chrom/start"),
    ("projection_quality", pb_projection_quality, canonical_pb_quality, "Projection including quality"),
    ("info_projection", pb_info_projection, canonical_pb_info, "Projected INFO field"),
    ("range_filter", pb_range_filter, canonical_pb_full, "Genomic range filter"),
    ("variant_lookup", pb_variant_lookup, canonical_pb_full, "Exact variant lookup"),
    ("sample_format", pb_sample_format, canonical_pb_sample, "FORMAT field for selected samples"),
    ("count_by_chrom", pb_count_by_chrom, canonical_count_map, "Simple aggregation"),
]

FIXED_SCENARIOS: dict[str, list[tuple[str, Callable[[], Any], Callable[[Any], dict[str, Any]], str]]] = {
    "zarr-python": [
        ("open_metadata", zarr_open_metadata, canonical_metadata, "Raw Zarr metadata"),
        ("full_core_scan", zarr_full_core, canonical_raw_full, "Raw core arrays"),
        ("projection_chrom_start", zarr_projection_min, canonical_raw_min, "Raw contig/position arrays"),
        ("projection_quality", zarr_projection_quality, canonical_raw_quality, "Raw quality array"),
        ("info_projection", zarr_info_projection, canonical_raw_info, "Raw INFO array"),
        ("range_filter", zarr_range_filter, canonical_raw_full, "Raw-array range equivalent"),
        ("variant_lookup", zarr_variant_lookup, canonical_raw_full, "Raw-array exact lookup"),
        ("sample_format", zarr_sample_format, canonical_raw_sample, "Raw FORMAT array slice over the same region and samples"),
        ("count_by_chrom", zarr_count_by_chrom, canonical_count_map, "Raw contig aggregation"),
    ],
    "sgkit": [
        ("open_metadata", sgkit_open_metadata, canonical_metadata, "sgkit/xarray dataset metadata"),
        ("full_core_scan", sgkit_full_core, canonical_sgkit_full, "Dataset core variables"),
        ("projection_chrom_start", sgkit_projection_min, canonical_sgkit_min, "Dataset contig/position variables"),
        ("projection_quality", sgkit_projection_quality, canonical_sgkit_quality, "Dataset quality variable"),
        ("info_projection", sgkit_info_projection, canonical_sgkit_info, "Dataset INFO variable"),
        ("range_filter", sgkit_range_filter, canonical_sgkit_full, "Dataset contig+position range filter"),
        ("variant_lookup", sgkit_variant_lookup, canonical_sgkit_full, "Dataset exact contig+position lookup"),
        ("sample_format", sgkit_sample_format, canonical_sgkit_sample, "Dataset call field over the same region and samples"),
        ("count_by_chrom", sgkit_count_by_chrom, canonical_count_map, "Dataset contig variable load"),
    ],
}

rows: list[ResultRow] = []

for partition in TARGET_PARTITIONS:
    for scenario, fn, canonicalize, notes in POLARS_SCENARIOS:
        rows.extend(
            measure(
                "polars-bio",
                scenario,
                str(partition),
                lambda fn=fn, partition=partition: fn(partition),
                canonicalize,
                notes=notes,
            )
        )

for tool, scenarios in FIXED_SCENARIOS.items():
    for scenario, fn, canonicalize, notes in scenarios:
        rows.extend(measure(tool, scenario, "fixed", fn, canonicalize, notes=notes))

raw_results_pd = pd.DataFrame([row.__dict__ for row in rows])
raw_results_pl = pl.from_pandas(raw_results_pd)

display(raw_results_pd)


## Result Tables

In [ ]:

def assert_comparable_results(raw: pd.DataFrame) -> pd.DataFrame:
    comparable = raw[raw["status"] == "ok"].copy()
    if comparable.empty:
        raise RuntimeError("No successful benchmark rows available for semantic comparison")
    rows = []
    mismatches = []
    for scenario, group in comparable.groupby("scenario", sort=True):
        unique = group[["tool", "partition", "canonical"]].drop_duplicates()
        canonical_count = unique["canonical"].nunique(dropna=False)
        tools = sorted(group["tool"].unique())
        rows.append({"scenario": scenario, "tools": ",".join(tools), "canonical_results": canonical_count})
        if canonical_count != 1:
            mismatches.append((scenario, unique))
    if mismatches:
        details = []
        for scenario, unique in mismatches:
            details.append(f"Scenario {scenario!r} returned {unique['canonical'].nunique()} semantic results:")
            details.append(unique.to_string(index=False))
        raise AssertionError("\n".join(details))
    return pd.DataFrame(rows)


comparison_pd = assert_comparable_results(raw_results_pd)
display(comparison_pd)


def summarize_results(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()
    ok = raw[raw["status"] == "ok"]
    if ok.empty:
        return pd.DataFrame(columns=["tool", "scenario", "partition", "median_s", "min_s", "max_s", "runs", "failures"])
    summary = (
        ok.groupby(["tool", "scenario", "partition"], dropna=False)
        .agg(
            median_s=("elapsed_s", "median"),
            min_s=("elapsed_s", "min"),
            max_s=("elapsed_s", "max"),
            runs=("elapsed_s", "count"),
        )
        .reset_index()
    )
    failures = (
        raw[raw["status"] != "ok"]
        .groupby(["tool", "scenario", "partition"], dropna=False)
        .size()
        .reset_index(name="failures")
    )
    summary = summary.merge(failures, on=["tool", "scenario", "partition"], how="left")
    summary["failures"] = summary["failures"].fillna(0).astype(int)
    return summary.sort_values(["scenario", "tool", "partition"]).reset_index(drop=True)


summary_pd = summarize_results(raw_results_pd)
summary_pl = pl.from_pandas(summary_pd) if not summary_pd.empty else pl.DataFrame()
failed_or_skipped_pd = raw_results_pd[raw_results_pd["status"] != "ok"].copy()

display(summary_pd)
if not failed_or_skipped_pd.empty:
    display(failed_or_skipped_pd[["tool", "scenario", "partition", "status", "error", "notes"]])


## Plots

In [ ]:
def plot_results(summary: pd.DataFrame) -> None:
    if summary.empty:
        print("No successful benchmark rows to plot.")
        return
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except Exception as exc:
        print(f"Plotting dependencies unavailable: {exc}")
        return

    plt.figure(figsize=(14, 6))
    plot_df = summary.copy()
    plot_df["tool_partition"] = plot_df["tool"] + " / " + plot_df["partition"].astype(str)
    sns.barplot(data=plot_df, x="scenario", y="median_s", hue="tool_partition")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Median runtime (s, log scale)")
    plt.title("VCF Zarr benchmark scenario runtime")
    plt.tight_layout()
    plt.show()

    pb_df = summary[summary["tool"] == "polars-bio"].copy()
    if pb_df.empty:
        print("No polars-bio rows available for scalability plot.")
        return
    pb_df["partition_int"] = pb_df["partition"].astype(int)
    base = pb_df.sort_values("partition_int").groupby("scenario").first()["median_s"].rename("base_s")
    pb_df = pb_df.merge(base, on="scenario")
    pb_df["speedup"] = pb_df["base_s"] / pb_df["median_s"]

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=pb_df, x="partition_int", y="speedup", hue="scenario", marker="o")
    plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
    plt.xlabel("datafusion.execution.target_partitions")
    plt.ylabel("Speedup vs smallest configured partition count")
    plt.title("polars-bio VCF Zarr partition scalability")
    plt.tight_layout()
    plt.show()


plot_results(summary_pd)

## Polars Result Tables

In [ ]:
raw_results_pl

In [ ]:
summary_pl